# Analyze the idQ simulation study

Run this notebook in your Great Lakes Jupyter session. Change **STUDY_NAME** in
the first code cell, then choose **Restart Kernel and Run All Cells**. Use the directory containing
`manifest.json`, such as `paper_final`, `paper_final_v014`, or `bernoulli_large_pilot`.

This notebook calls the existing idQ analyzer using your project's Python
environment. It reads the simulation outputs; it does not rerun any SAT problems
or change package source. The raw CSVs, metadata sidecars, and manifest must stay
together. The default requires a completed study. For monitoring unfinished jobs,
set `ALLOW_PARTIAL=True`; those summaries are explicitly marked as interim.

The notebook works with the original paper study and the separate K=20,30
Bernoulli extension. Tables display the K values recorded in the selected study, followed by optional plots
and an optional view of individual simulation records. Percentages are on the
0–100 scale; times are seconds per matrix.

Your Jupyter kernel needs pandas. If the import fails, run `%pip install pandas`
in a notebook cell and restart the kernel. The optional plots need matplotlib.


In [10]:
from pathlib import Path
import json
import os
import subprocess
import sys
import pandas as pd

PROJECT = Path.home() / "idQ"
STUDY_NAME = "bernoulli_large_pilot"  # Or paper_final, paper_final_v014, bernoulli_large_full.
ALLOW_PARTIAL = True
SHOW_PLOTS = True
LOAD_RAW = False  # Optional individual-matrix records, below.

PROJECT = PROJECT.expanduser().resolve()
STUDY = PROJECT / "data" / STUDY_NAME
# For data on scratch storage, replace STUDY with its absolute directory.

candidate_pythons = [PROJECT / ".venv-idq/bin/python", PROJECT / ".venv/bin/python"]
RUN_PYTHON = next((p for p in candidate_pythons if p.is_file()), Path(sys.executable))
# If you use another environment, set RUN_PYTHON to its Python executable.

if not (PROJECT / "src/idQ/experiments/paper_summary.py").is_file():
    raise FileNotFoundError("Set PROJECT to the updated idQ package directory.")
if not (STUDY / "manifest.json").is_file():
    available = sorted(p.parent.name for p in (PROJECT / "data").glob("*/manifest.json"))
    raise FileNotFoundError(f"No manifest.json in {STUDY}. Available study folders: {available}")

print("Study:", STUDY)
print("Analysis Python:", RUN_PYTHON)
print("Jupyter Python:", sys.executable)


Study: /home/lemonkey/idQ/data/bernoulli_large_pilot
Analysis Python: /home/lemonkey/idQ/.venv-idq/bin/python
Jupyter Python: /sw/pkgs/arc/python3.10-anaconda/2023.03/bin/python


## 1. Validate and aggregate

This invokes the same analysis as the automatic Slurm analysis job. It checks
the expected settings and replicate counts, duplicate records, matrix indicators,
solver settings, and provenance before producing summaries. Running it again
refreshes the derived outputs; raw data are not modified.

If the study is incomplete, this cell stops with an explanation. Set
`ALLOW_PARTIAL=True` only to inspect progress, then rerun from the first cell.
Incomplete results can preferentially omit difficult SAT instances and should
not be used as final simulation estimates.

When switching studies, restart the kernel and run every cell from the top.
Editing `STUDY_NAME` alone does not reload an existing DataFrame or replace old
cell outputs. If aggregation stops, later saved outputs are not new results.


In [11]:
summary = None
cells = None
bernoulli = None
sparse = None
env = os.environ.copy()
env["PYTHONPATH"] = str(PROJECT / "src") + (os.pathsep + env["PYTHONPATH"] if env.get("PYTHONPATH") else "")
command = [str(RUN_PYTHON), "-m", "idQ.experiments.paper_summary", "--study-dir", str(STUDY)]
if ALLOW_PARTIAL:
    command.append("--allow-partial")

print("Validating and aggregating:", STUDY)
run = subprocess.run(command, cwd=str(PROJECT), env=env, capture_output=True, text=True)
if run.returncode:
    print(run.stdout)
    print(run.stderr)
    raise RuntimeError("Analysis stopped. Resolve the reported issue before using the summaries.")

report = json.loads(run.stdout.strip().splitlines()[-1])
status = report["status"]
suffix = {"partial": "_PARTIAL", "smoke_complete": "_SMOKE"}.get(status, "")
ANALYSIS = STUDY / ("analysis" + suffix.lower())
summary = json.loads((ANALYSIS / "summary.json").read_text())
runtime_environment = json.loads((ANALYSIS / "runtime_environment.json").read_text())
manifest = json.loads((STUDY / "manifest.json").read_text())
cells = pd.DataFrame(summary["cells"])
for name in ("J", "K", "n", "expected_n", "m"):
    cells[name] = cells[name].astype("Int64")
bernoulli = cells.loc[cells["design"].eq("bernoulli")].copy()
sparse = cells.loc[cells["design"].eq("row_sparsity")].copy()

print("Status:", status)
print(f"Validated {summary['actual_total']:,} / {summary['expected_total']:,} matrices.")
print("Output directory:", ANALYSIS)
if status == "partial":
    print("INTERIM RESULTS: missing or interrupted tasks remain unresolved.")
elif status == "smoke_complete":
    print("PILOT / TEST STUDY: these are not the full 1,000-replicate results.")


Validating and aggregating: /home/lemonkey/idQ/data/bernoulli_large_pilot
Status: smoke_complete
Validated 360 / 360 matrices.
Output directory: /home/lemonkey/idQ/data/bernoulli_large_pilot/analysis_smoke
PILOT / TEST STUDY: these are not the full 1,000-replicate results.


In [12]:
# Counts are read from this study's newly generated summary.
print("Loaded study:", STUDY)
print("Summary:", ANALYSIS / "summary.json")
print("Reported profile:", summary.get("study_profile", "not supplied"))
print("Bernoulli K values:", sorted(int(k) for k in bernoulli["K"].dropna().unique()))
print("Completed and expected matrices by design and K:")
coverage_by_k = cells.groupby(["design", "K"])[["n", "expected_n"]].sum()
coverage_by_k["missing"] = coverage_by_k["expected_n"] - coverage_by_k["n"]
display(coverage_by_k)

# A full study has 1,000 observations per setting; the suggested pilot has 10.
coverage = cells[["design", "J", "K", "p", "m", "n", "expected_n"]].copy()
coverage["missing"] = coverage["expected_n"] - coverage["n"]
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(coverage)


Loaded study: /home/lemonkey/idQ/data/bernoulli_large_pilot
Summary: /home/lemonkey/idQ/data/bernoulli_large_pilot/analysis_smoke/summary.json
Reported profile: not supplied
Bernoulli K values: [5, 10]
Completed and expected matrices by design and K:


n  expected_n  missing
design       K                           
bernoulli    5   150         150        0
             10  150         150        0
row_sparsity 10   60          60        0

,design,J,K,p,m,n,expected_n,missing
0,bernoulli,25,5,0.1,<NA>,10,10,0
1,bernoulli,25,5,0.3,<NA>,10,10,0
2,bernoulli,25,5,0.5,<NA>,10,10,0
3,bernoulli,25,5,0.7,<NA>,10,10,0
4,bernoulli,25,5,0.9,<NA>,10,10,0
5,bernoulli,25,10,0.1,<NA>,10,10,0
6,bernoulli,25,10,0.3,<NA>,10,10,0
7,bernoulli,25,10,0.5,<NA>,10,10,0
8,bernoulli,25,10,0.7,<NA>,10,10,0
9,bernoulli,25,10,0.9,<NA>,10,10,0


## 2. Bernoulli proportions — the first paper table

Each entry is a percentage over all matrices in that setting. The last panel
is the conjunction of the two-column check and the three-column check.


In [4]:
def bernoulli_panel(column, title):
    print(title)
    table = bernoulli.pivot(index="p", columns=["J", "K"], values=column).sort_index(axis=1)
    with pd.option_context("display.max_columns", None):
        display(table.round(1))
    return table

bern_tables = {}
for column, title in [
    ("pct_identifiable", "Identifiable (%)"),
    ("pct_complete", "Complete (%)"),
    ("pct_two_column", "Passes the two-column condition (%)"),
    ("pct_both_necessary", "Passes both necessary submatrix checks (%)"),
]:
    bern_tables[column] = bernoulli_panel(column, title)


Identifiable (%)


J      25           50            100       
K       5     10     5      10     5      10
p                                           
0.1   80.0   0.0   90.0   80.0  100.0  100.0
0.3   70.0  90.0  100.0  100.0  100.0  100.0
0.5  100.0  80.0  100.0  100.0  100.0  100.0
0.7   90.0   0.0  100.0   90.0  100.0  100.0
0.9    0.0   0.0    0.0    0.0   10.0    0.0

Complete (%)


J     25         50           100       
K      5    10    5     10     5      10
p                                       
0.1  40.0  0.0  90.0  20.0  100.0  100.0
0.3  50.0  0.0  80.0   0.0  100.0   10.0
0.5   0.0  0.0  20.0   0.0  100.0    0.0
0.7   0.0  0.0   0.0   0.0    0.0    0.0
0.9   0.0  0.0   0.0   0.0    0.0    0.0

Passes the two-column condition (%)


J      25           50            100       
K       5     10     5      10     5      10
p                                           
0.1   80.0   0.0   90.0   80.0  100.0  100.0
0.3   70.0  90.0  100.0  100.0  100.0  100.0
0.5  100.0  80.0  100.0  100.0  100.0  100.0
0.7  100.0  70.0  100.0  100.0  100.0  100.0
0.9   40.0   0.0  100.0   70.0  100.0  100.0

Passes both necessary submatrix checks (%)


J      25           50            100       
K       5     10     5      10     5      10
p                                           
0.1   80.0   0.0   90.0   80.0  100.0  100.0
0.3   70.0  90.0  100.0  100.0  100.0  100.0
0.5  100.0  80.0  100.0  100.0  100.0  100.0
0.7  100.0  40.0  100.0  100.0  100.0  100.0
0.9    0.0   0.0   10.0    0.0   40.0   10.0

## 3. Incompleteness and no pure nodes among identifiable matrices

These are **conditional percentages**, with the number of identifiable matrices
as the denominator. NaN means that denominator is zero (or that a cell has no
completed observations in an interim analysis); it is not a zero percentage.
The saved LaTeX tables display these undefined entries as dashes.


In [5]:
conditional_tables = {}
for column, title in [
    ("pct_incomplete_given_identifiable", "Incomplete, among identifiable matrices (%)"),
    ("pct_no_pure_given_identifiable", "No pure nodes, among identifiable matrices (%)"),
]:
    conditional_tables[column] = bernoulli_panel(column, title)

print("Identifiable denominator in each setting")
display(bernoulli.pivot(index="p", columns=["J", "K"],
                        values="conditional_denominator_identifiable").sort_index(axis=1))


Incomplete, among identifiable matrices (%)


J      25            50            100       
K       5      10     5      10     5      10
p                                            
0.1   50.0    NaN    0.0   75.0    0.0    0.0
0.3   28.6  100.0   20.0  100.0    0.0   90.0
0.5  100.0  100.0   80.0  100.0    0.0  100.0
0.7  100.0    NaN  100.0  100.0  100.0  100.0
0.9    NaN    NaN    NaN    NaN  100.0    NaN

No pure nodes, among identifiable matrices (%)


J     25           50            100      
K      5      10    5      10     5     10
p                                         
0.1   0.0    NaN   0.0    0.0    0.0   0.0
0.3   0.0    0.0   0.0    0.0    0.0   0.0
0.5   0.0  100.0   0.0   50.0    0.0  50.0
0.7  33.3    NaN  10.0  100.0    0.0  90.0
0.9   NaN    NaN   NaN    NaN  100.0   NaN

Identifiable denominator in each setting


J   25     50      100    
K    5  10  5   10  5   10
p                         
0.1   8  0   9   8  10  10
0.3   7  9  10  10  10  10
0.5  10  8  10  10  10  10
0.7   9  0  10   9  10  10
0.9   0  0   0   0   1   0

## 4. Sparsity proportions

This section displays the third paper table. It is skipped for a Bernoulli-only
K=20,30 study.


In [30]:
sparse_tables = {}
if sparse.empty:
    print("This study contains Bernoulli samples only.")
else:
    for column, title in [
        ("pct_identifiable", "Identifiable (%)"),
        ("pct_incomplete_given_identifiable", "Incomplete, among identifiable matrices (%)"),
        ("pct_two_column", "Passes the two-column condition (%)"),
        ("pct_both_necessary", "Passes both necessary submatrix checks (%)"),
    ]:
        print(title)
        table = sparse.pivot(index="J", columns="m", values=column).sort_index(axis=1)
        sparse_tables[column] = table
        display(table.round(1))


Identifiable (%)


m,3,4
J,,
25,60.0,70.0
50,100.0,100.0
100,100.0,100.0


Incomplete, among identifiable matrices (%)


m,3,4
J,,
25,100.0,100.0
50,100.0,90.0
100,30.0,60.0


Passes the two-column condition (%)


m,3,4
J,,
25,60.0,70.0
50,100.0,100.0
100,100.0,100.0


Passes both necessary submatrix checks (%)


m,3,4
J,,
25,60.0,70.0
50,100.0,100.0
100,100.0,100.0


## 5. Runtime and basis reduction

This table displays every Bernoulli K value recorded in the selected study.
For the paper study, this includes K=5 and K=10; the paper's computational
table uses K=10. The extension includes K=20 and K=30. **All runtime means include every matrix in the setting**, including
those resolved by preprocessing. They are not conditional on invoking SAT.

- `mean_preprocess_time`: basis reduction plus the checks actually executed.
- `mean_algorithm_time`: all timed algorithm stages, including CNF construction,
  solver setup/solving, and witness checking/lifting where applicable.
- `mean_sat_time`: time in the SAT path, including construction and checking;
  zero contribution from a matrix that did not invoke SAT. This is not
  solver-only time.

Sampling, extra simulation diagnostics, metadata collection, and file writing
are excluded from the algorithm time. Keep the unrounded values for calculations;
formatting the notebook tables does not change the data.


In [ ]:
# Use the data itself, including when study_profile is absent or outdated.
runtime_ks = sorted(int(k) for k in bernoulli["K"].dropna().unique())
print("Runtime table for:", STUDY)
print("Displayed K values:", runtime_ks)
runtime_table = bernoulli.loc[bernoulli["K"].isin(runtime_ks), [
    "J", "K", "p", "n", "mean_J_basis", "pct_sat_called",
    "mean_preprocess_time", "mean_algorithm_time", "mean_sat_time",
]].sort_values(["K", "J", "p"])
with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.float_format", "{:.6g}".format):
    display(runtime_table)

print("Basis reduction across all settings")
basis_table = cells[["design", "J", "K", "p", "m", "mean_J_basis"]].copy()
basis_table["mean_rows_removed_pct"] = 100 * (1 - basis_table["mean_J_basis"] / basis_table["J"])
display(basis_table.round(2))


## 6. Processor and software information

Use this information when describing computational results. Different processor
models may have different runtimes. Changes from earlier results also reflect
new random matrices unless the original inputs were reused.


In [ ]:
print("Software versions:")
display(pd.Series(runtime_environment.get("software_versions") or {}, name="version"))
print("Processor models and number of completed jobs:")
display(pd.Series(runtime_environment.get("cpu_model_job_counts", {}), name="jobs"))
print("Mixed processor models:", runtime_environment.get("mixed_hardware"))
print("Recorded timing definitions:")
for name, definition in (runtime_environment.get("timing_definitions") or {}).items():
    print(f"{name}: {definition}")


## 7. Optional plots

The upper panels show identifiability percentages; lower panels show mean total
algorithm time on a logarithmic scale. These use the same summaries as the tables.
Set `SHOW_PLOTS=False` in the first cell to skip them. If matplotlib is missing,
run `%pip install matplotlib` in a notebook cell and rerun this section.


In [ ]:
fig = None
if SHOW_PLOTS and bernoulli["n"].sum() > 0:
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        print("Optional plots need matplotlib: run %pip install matplotlib in this notebook.")
    else:
        ks = sorted(bernoulli["K"].dropna().unique())
        fig, axes = plt.subplots(2, len(ks), figsize=(5 * len(ks), 7), squeeze=False,
                                 sharex=True, constrained_layout=True)
        for col, K in enumerate(ks):
            for J, group in bernoulli.loc[bernoulli["K"].eq(K)].groupby("J"):
                group = group.sort_values("p")
                axes[0, col].plot(group["p"], group["pct_identifiable"], marker="o", label=f"J={J}")
                axes[1, col].plot(group["p"], group["mean_algorithm_time"], marker="o", label=f"J={J}")
            axes[0, col].set(title=f"K={K}", ylabel="Identifiable (%)", ylim=(-2, 102))
            axes[1, col].set(xlabel="Bernoulli probability p", ylabel="Mean total algorithm time (s)")
            if (bernoulli.loc[bernoulli["K"].eq(K), "mean_algorithm_time"] > 0).any():
                axes[1, col].set_yscale("log")
            for row in range(2):
                axes[row, col].grid(alpha=0.25)
                axes[row, col].legend()
        fig.suptitle(f"{STUDY_NAME} — {status}")
        plt.show()
        # To export after reviewing: fig.savefig(ANALYSIS / "bernoulli_overview.pdf")


## 8. Optional individual-matrix records

Set `LOAD_RAW=True` in the first cell, then rerun. This loads selected columns
only from tasks accepted by the analyzer, avoiding duplicate/partial files and
unrelated studies. `raw` then contains one row per matrix for your own exploration.

The runtime distribution below is exploratory; its median or SAT-conditional
mean should not replace the all-matrix mean in the paper's runtime table.


In [ ]:
raw = None
if LOAD_RAW:
    accepted_ids = {str(job["task_id"]) for job in runtime_environment["jobs"]}
    selected_columns = {
        "J", "K", "p", "m_requested", "design", "N", "seed", "sim",
        "identifiable", "branch", "branch_label", "sat_called", "J_basis",
        "identity_original", "no_pure_nodes_original", "two_column_pass_original", "I_3c",
        "basis_time", "preprocess_time", "sat_time", "algorithm_time",
    }
    frames = []
    for task in manifest["tasks"]:
        if str(task["task_id"]) not in accepted_ids:
            continue
        frame = pd.read_csv(STUDY / task["csv_path"], usecols=lambda name: name in selected_columns)
        frame["task_id"] = task["task_id"]
        frames.append(frame)
    raw = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    assert len(raw) == summary["actual_total"]
    print(f"Loaded {len(raw):,} validated matrix records.")
    display(raw.head())
    if not raw.empty:
        raw_bern = raw.loc[raw["design"].eq("bernoulli")]
        display(raw_bern.groupby(["J", "K", "p"])["algorithm_time"].agg(
            count="count", mean="mean", median="median", maximum="max"))
else:
    print("Set LOAD_RAW=True in the first cell to load individual-matrix records.")


## 9. Files for the manuscript

The analyzer already saved the LaTeX tables, exact counts and unrounded summaries,
and runtime provenance. Replace the corresponding manuscript tables and update
numerical statements from the new results. Preserve the complete study folder
so its inputs and measurements remain reproducible.

To analyze the separate extension, change `STUDY_NAME` to
`bernoulli_large_full` (or `bernoulli_large_pilot`) and run the notebook again.
Keep each study in its own directory.


In [ ]:
print("Analysis files:")
for path in sorted(ANALYSIS.iterdir()):
    if path.is_file():
        print(path)
# Report existing table files instead of guessing their name from study_profile.
LATEX_TABLES = [path for path in sorted(ANALYSIS.glob(f"*tables{suffix}.tex")) if path.is_file()]
if LATEX_TABLES:
    print("\nLaTeX tables:")
    for path in LATEX_TABLES:
        print(path)
else:
    print("No matching LaTeX table file found; check the analyzer output above.")
print("Exact counts and unrounded means:", ANALYSIS / "cell_summary.csv")
